# 09 - DBLP Scraper: Conference/Year -> Main Volume -> Papers

This notebook fetches, for each conference key + year, the main track proceedings
(filtering out Short Papers / Findings / Workshops / Demos / Tutorials), then
extracts every paper (title, authors, pages) into a flat CSV ready for
OpenAlex title-matching (feeds into your nb08 matching pipeline).

**Note:** Requires internet access. Run this locally / Colab / wherever
`requests` can reach dblp.org. Respects DBLP's crawl etiquette (1.5s delay,
retry on 429).

In [1]:
import requests
import time
import re
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {'User-Agent': 'Mozilla/5.0 (research script; contact: sherotowshaw@gmail.com)'}
SLEEP_SECONDS = 3.0
TIMEOUT = 30

session = requests.Session()
session.headers.update(HEADERS)

def fetch(url, retries=6):
    global session
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=TIMEOUT)
        except requests.exceptions.RequestException as e:
            wait = 5 * (attempt + 1)
            print(f'Attempt {attempt+1} failed for {url}: {e!r} -- retrying in {wait}s')
            session.close()
            session = requests.Session()
            session.headers.update(HEADERS)
            time.sleep(wait)
            continue

        if resp.status_code == 200:
            time.sleep(SLEEP_SECONDS)
            return resp.text
        elif resp.status_code == 429:
            wait = int(resp.headers.get('Retry-After', 10))
            print(f'429 rate limited on {url}, waiting {wait}s...')
            time.sleep(wait)
        elif resp.status_code == 503:
            wait = 15 * (attempt + 1)
            print(f'503 (likely bot-blocked) on {url}, backing off {wait}s...')
            time.sleep(wait)
        else:
            print(f'Failed {url}: status {resp.status_code}')
            time.sleep(SLEEP_SECONDS)
            return None
    print(f'Giving up on {url} after {retries} attempts')
    return None


## Step 1: Conference keys to scrape

Edit this list with your 31 conference DBLP keys.

In [2]:
CONFERENCE_KEYS = [
    'aaai', 'acl', 'chi', 'cikm', 'cvpr', 'focs', 'fse', 'iccv', 'icml',
    'icse', 'icwsm', 'ijcai', 'infocom', 'jcdl', 'kdd', 'mobicom', 'nips',
    'osdi', 'pldi', 'pods', 'sp', 'sigcomm', 'sigir', 'sigmetrics', 'sigmod',
    'soda', 'sosp', 'stoc', 'uist', 'vldb', 'www'
]
print(f'{len(CONFERENCE_KEYS)} conference keys loaded')


31 conference keys loaded


## Year range filter

Your comparison target is award-winning papers from **2000-2018**, not the
full DBLP history. Restricting to this range cuts request volume dramatically
(fewer years = fewer contents-page fetches = less risk of getting blocked
again) and keeps the dataset focused on what you actually need to match
against.

In [3]:
YEAR_START = 2000
YEAR_END = 2018

def in_year_range(year_str):
    try:
        y = int(year_str)
    except (TypeError, ValueError):
        return False
    return YEAR_START <= y <= YEAR_END


## Step 2: Parse the venue index page -> list of (year, title, contents_url) per volume

In [4]:
EXCLUDE_KEYWORDS = [
    'short papers', 'system demonstrations', 'demo track', 'demonstrations',
    'student research', 'tutorial abstracts', 'industry track',
    'findings of', 'workshop', 'companion'
]

def parse_index_page(html, conf_key):
    soup = BeautifulSoup(html, 'html.parser')
    volumes = []
    for h in soup.select('header.h2'):
        year_tag = h.find('h2')
        if not year_tag or not year_tag.get('id'):
            continue
        year_id = year_tag['id']
        year_match = re.search(r'(19|20)\d{2}', year_id)
        if not year_match:
            continue
        year = year_match.group(0)
        ul = h.find_next_sibling('ul', class_='publ-list')
        if not ul:
            continue
        for li in ul.select('li.entry.editor.toc'):
            title_span = li.select_one('span.title')
            toc_link = li.select_one('a.toc-link')
            if title_span and toc_link:
                volumes.append({
                    'conference': conf_key,
                    'year': year,
                    'title': title_span.get_text(strip=True),
                    'contents_url': toc_link['href']
                })
    return volumes

def pick_main_volume(volumes_for_year):
    candidates = [
        v for v in volumes_for_year
        if not any(kw in v['title'].lower() for kw in EXCLUDE_KEYWORDS)
    ]
    return candidates[0] if candidates else None

def get_main_volumes_for_conference(conf_key):
    url = f'https://dblp.org/db/conf/{conf_key}/index.html'
    html = fetch(url)
    if html is None:
        return []
    all_volumes = parse_index_page(html, conf_key)
    all_volumes = [v for v in all_volumes if in_year_range(v['year'])]
    by_year = {}
    for v in all_volumes:
        by_year.setdefault(v['year'], []).append(v)
    main_volumes = []
    for year, vols in by_year.items():
        main = pick_main_volume(vols)
        if main:
            main_volumes.append(main)
        else:
            print(f'  WARNING: no main volume found for {conf_key} {year}')
    return sorted(main_volumes, key=lambda v: v['year'])


## Step 3: Test on a single conference before scaling up

In [5]:
test_volumes = get_main_volumes_for_conference('acl')
df_test = pd.DataFrame(test_volumes)
df_test.tail(10)


,conference,year,title,contents_url
9,acl,2009,"ACL 2009, Proceedings of the 47th Annual Meeti...",https://dblp.org/db/conf/acl/acl2009.html
10,acl,2010,"ACL 2010, Proceedings of the 48th Annual Meeti...",https://dblp.org/db/conf/acl/acl2010.html
11,acl,2011,The 49th Annual Meeting of the Association for...,https://dblp.org/db/conf/acl/acl2011.html
12,acl,2012,The 50th Annual Meeting of the Association for...,https://dblp.org/db/conf/acl/acl2012-1.html
13,acl,2013,Proceedings of the 51st Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2013-1.html
14,acl,2014,Proceedings of the 52nd Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2014-1.html
15,acl,2015,Proceedings of the 53rd Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2015-1.html
16,acl,2016,Proceedings of the 54th Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2016-1.html
17,acl,2017,Proceedings of the 55th Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2017-1.html
18,acl,2018,Proceedings of the 56th Annual Meeting of the ...,https://dblp.org/db/conf/acl/acl2018-1.html


## Step 4: Parse a contents page -> individual papers

In [6]:
def parse_contents_page(html, conf_key, year):
    soup = BeautifulSoup(html, 'html.parser')
    papers = []
    for li in soup.select('li.entry.inproceedings, li.entry.article'):
        title_span = li.select_one('span.title')
        if not title_span:
            continue
        title = title_span.get_text(strip=True)
        authors = [a.get_text(strip=True) for a in li.select('span[itemprop="author"] span[itemprop="name"]')]
        pages_tag = li.select_one('span[itemprop="pagination"]')
        pages = pages_tag.get_text(strip=True) if pages_tag else None
        papers.append({
            'conference': conf_key,
            'year': year,
            'title': title,
            'authors': '; '.join(authors),
            'pages': pages
        })
    return papers

def get_papers_for_volume(volume):
    html = fetch(volume['contents_url'])
    if html is None:
        return []
    return parse_contents_page(html, volume['conference'], volume['year'])


## Step 5: Test paper extraction on one volume

In [7]:
if len(test_volumes) > 0:
    sample_volume = test_volumes[-1]
    print('Testing on:', sample_volume['title'], sample_volume['contents_url'])
    sample_papers = get_papers_for_volume(sample_volume)
    df_sample = pd.DataFrame(sample_papers)
    print(f'{len(df_sample)} papers found')
    df_sample.head(10)


Testing on: Proceedings of the 56th Annual Meeting of the Association for Computational Linguistics, ACL 2018, Melbourne, Australia, July 15-20, 2018, Volume 1: Long Papers. https://dblp.org/db/conf/acl/acl2018-1.html
256 papers found


## Step 6: Full pipeline across all conferences

WARNING: This will make many requests (1 index page + N contents pages per
conference). With 31 conferences x ~20-30 years each, expect 600-1000+
requests total. At 1.5s delay each, budget ~20-30 minutes. Consider running
in batches / saving intermediate progress.

In [8]:
import os
os.makedirs('output', exist_ok=True)

all_papers = []
all_volumes_log = []
completed_conferences = []

for conf_key in CONFERENCE_KEYS:
    print(f'--- Processing {conf_key} ---')
    volumes = get_main_volumes_for_conference(conf_key)
    all_volumes_log.extend(volumes)
    conf_papers = []
    for vol in volumes:
        papers = get_papers_for_volume(vol)
        conf_papers.extend(papers)
        print(f'  {vol["year"]}: {len(papers)} papers')
    all_papers.extend(conf_papers)
    completed_conferences.append(conf_key)

    # Checkpoint after every conference so progress survives a crash/block
    pd.DataFrame(all_papers).to_csv('output/dblp_papers_raw.csv', index=False)
    pd.DataFrame(all_volumes_log).to_csv('output/dblp_main_volumes_log.csv', index=False)
    print(f'  Checkpoint saved. Completed so far: {completed_conferences}')

df_papers = pd.DataFrame(all_papers)
df_volumes_log = pd.DataFrame(all_volumes_log)
print(f'\nTOTAL: {len(df_papers)} papers across {len(df_volumes_log)} conference-years')


--- Processing aaai ---
  2000: 233 papers
  2002: 180 papers
503 (likely bot-blocked) on https://dblp.org/db/conf/aaai/aaai2004.html, backing off 15s...
  2004: 194 papers
  2005: 325 papers
  2006: 385 papers
  2007: 368 papers
503 (likely bot-blocked) on https://dblp.org/db/conf/aaai/aaai2008.html, backing off 15s...
  2008: 356 papers
  2010: 313 papers
  2011: 319 papers
  2012: 353 papers
  2013: 251 papers
  2014: 474 papers
  2015: 674 papers
  2016: 691 papers
  2017: 786 papers
  2018: 1102 papers
  Checkpoint saved. Completed so far: ['aaai']
--- Processing acl ---
  2000: 79 papers
  2001: 70 papers
  2002: 65 papers
503 (likely bot-blocked) on https://dblp.org/db/conf/acl/acl2003.html, backing off 15s...
  2003: 71 papers
  2004: 88 papers
  2005: 134 papers
  2006: 307 papers
503 (likely bot-blocked) on https://dblp.org/db/conf/acl/acl2007.html, backing off 15s...
  2007: 204 papers
  2008: 119 papers
  2009: 121 papers
  2010: 160 papers
  2011: 164 papers
  2012: 111 pa

## Step 6b: Resume after a crash/block

If Step 6 got interrupted partway (e.g. blocked mid-conference), re-run with
only the remaining conference keys. Check `output/dblp_main_volumes_log.csv`
for which conferences already have data, then set `CONFERENCE_KEYS` to the
remainder and re-run Step 6.

## Step 7: Save outputs

In [9]:
import os
os.makedirs('output', exist_ok=True)
df_papers.to_csv('output/dblp_papers_raw.csv', index=False)
df_volumes_log.to_csv('output/dblp_main_volumes_log.csv', index=False)
print('Saved output/dblp_papers_raw.csv and output/dblp_main_volumes_log.csv')


Saved output/dblp_papers_raw.csv and output/dblp_main_volumes_log.csv
